# 📚 AI Research Paper Intelligence System — CBSOT Summer Internship 2026
## Notebook 2: Features Testing — Semantic Search | BART Summarization | KeyBERT | Paper Similarity Graph | Citation Network | Compare Papers | PDF Export

**Run Notebook 1 first to generate the FAISS index!**

---

## 1. Install & Import

In [ ]:
!pip install datasets sentence-transformers faiss-cpu transformers keybert torch networkx pyvis reportlab -q
print('Done!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import pickle
import os
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline
from keybert import KeyBERT
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

print('All libraries imported!')

## 2. Load Saved Index & Data

In [ ]:
# Load FAISS index
index = faiss.read_index('outputs/faiss_index/papers.index')

# Load dataframe
df = pd.read_pickle('outputs/data/papers_df.pkl')

# Load embeddings
embeddings = np.load('outputs/data/embeddings.npy')

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f'✅ Index loaded: {index.ntotal:,} papers')
print(f'✅ DataFrame  : {df.shape}')
print(f'✅ Embeddings : {embeddings.shape}')

## 3. Core Search Function (with MMR)

In [ ]:
def semantic_search(query, top_k=10, mmr=True, diversity=0.3):
    """Semantic search with optional MMR diversity re-ranking."""
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    # Get more candidates for MMR
    fetch_k = top_k * 3 if mmr else top_k
    scores, indices = index.search(q_emb, fetch_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        results.append({
            'idx': int(idx),
            'title': df['title'].iloc[idx],
            'abstract': df['abstract'].iloc[idx],
            'score': float(score)
        })

    if not mmr:
        return results[:top_k]

    # MMR — Maximal Marginal Relevance for diversity
    candidate_embeddings = np.array([embeddings[r['idx']] for r in results])
    selected = [0]
    while len(selected) < top_k:
        remaining = [i for i in range(len(results)) if i not in selected]
        if not remaining:
            break
        mmr_scores = []
        for i in remaining:
            relevance = results[i]['score']
            sim_to_selected = max(
                cosine_similarity(
                    candidate_embeddings[i].reshape(1, -1),
                    candidate_embeddings[j].reshape(1, -1)
                )[0][0] for j in selected
            )
            mmr_scores.append((1 - diversity) * relevance - diversity * sim_to_selected)
        selected.append(remaining[np.argmax(mmr_scores)])

    return [results[i] for i in selected]

# Test
results = semantic_search('graph neural networks node classification', top_k=5)
print('Top 5 results for "graph neural networks node classification":')
for i, r in enumerate(results):
    print(f'  {i+1}. [{r["score"]:.4f}] {r["title"]}')

## 4. BART Summarization

In [ ]:
print('Loading BART summarizer...')
summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6', device=-1)
print('BART loaded!')

def summarize(text, max_len=120, min_len=40):
    """Summarize text using DistilBART."""
    text = text[:1024]  # BART max input
    try:
        summary = summarizer(text, max_length=max_len, min_length=min_len,
                             do_sample=False, truncation=True)
        return summary[0]['summary_text']
    except:
        return text[:300] + '...'

# Test
sample = results[0]['abstract']
print(f'\nOriginal ({len(sample.split())} words):')
print(sample[:300])
print(f'\nSummary:')
print(summarize(sample))

## 5. KeyBERT Keyword Extraction

In [ ]:
kw_model = KeyBERT()

def extract_keywords(text, top_n=8):
    """Extract keywords using KeyBERT with MMR diversity."""
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        use_mmr=True,
        diversity=0.5,
        top_n=top_n
    )
    return [kw[0] for kw in keywords]

# Test
kws = extract_keywords(results[0]['abstract'])
print(f'Keywords for "{results[0]["title"][:60]}...":')
print(kws)

## 6. 🌐 Paper Similarity Graph (UNIQUE)
### NetworkX graph showing how papers relate to each other

In [ ]:
def build_similarity_graph(query, top_k=8, sim_threshold=0.75):
    """
    Build a similarity graph of papers.
    Nodes = papers, Edges = cosine similarity > threshold
    """
    results = semantic_search(query, top_k=top_k, mmr=False)
    paper_embeddings = np.array([embeddings[r['idx']] for r in results])

    # Compute pairwise similarity
    sim_matrix = cosine_similarity(paper_embeddings)

    G = nx.Graph()

    # Add nodes
    for i, r in enumerate(results):
        short_title = r['title'][:40] + '...' if len(r['title']) > 40 else r['title']
        G.add_node(i, title=short_title, score=r['score'], full_title=r['title'])

    # Add edges where similarity > threshold
    for i in range(len(results)):
        for j in range(i+1, len(results)):
            if sim_matrix[i][j] > sim_threshold:
                G.add_edge(i, j, weight=float(sim_matrix[i][j]))

    return G, results, sim_matrix


def plot_similarity_graph(query, top_k=8, sim_threshold=0.75):
    G, results, sim_matrix = build_similarity_graph(query, top_k, sim_threshold)

    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, seed=42, k=2)

    # Node colors by relevance score
    scores = [G.nodes[n]['score'] for n in G.nodes]
    node_colors = plt.cm.YlOrRd(np.array(scores) / max(scores))

    # Edge widths by similarity
    edge_weights = [G[u][v]['weight'] * 3 for u, v in G.edges]

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000, alpha=0.9)
    nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, edge_color='#3498db')

    labels = {n: G.nodes[n]['title'] for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels, font_size=7, font_weight='bold')

    # Edge labels (similarity scores)
    edge_labels = {(u, v): f"{G[u][v]['weight']:.2f}" for u, v in G.edges}
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=6, alpha=0.7)

    plt.title(f'Paper Similarity Graph\nQuery: "{query}"\n(Edge = cosine similarity > {sim_threshold})',
              fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('outputs/03_similarity_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    return G

# Test
G = plot_similarity_graph('attention mechanism transformer', top_k=8, sim_threshold=0.72)

## 7. 🕸️ Citation Network — Topic Clusters (UNIQUE)
### Shows how different topics/clusters of papers connect

In [ ]:
from sklearn.cluster import KMeans

def build_citation_network(queries, papers_per_query=5):
    """
    Build a citation-style network across multiple topics.
    Each query = a topic cluster. Papers within a cluster are connected,
    and cross-cluster edges show topic overlap.
    """
    all_papers = []
    topic_map = {}

    for topic_idx, query in enumerate(queries):
        results = semantic_search(query, top_k=papers_per_query, mmr=True)
        for r in results:
            r['topic'] = query
            r['topic_idx'] = topic_idx
            all_papers.append(r)

    G = nx.Graph()
    colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

    for i, p in enumerate(all_papers):
        short = p['title'][:35] + '...' if len(p['title']) > 35 else p['title']
        G.add_node(i, title=short, topic=p['topic'],
                   color=colors[p['topic_idx'] % len(colors)])

    # Intra-cluster edges
    paper_embeddings = np.array([embeddings[p['idx']] for p in all_papers])
    sim_matrix = cosine_similarity(paper_embeddings)

    for i in range(len(all_papers)):
        for j in range(i+1, len(all_papers)):
            same_topic = all_papers[i]['topic_idx'] == all_papers[j]['topic_idx']
            threshold = 0.70 if same_topic else 0.80  # stricter for cross-topic
            if sim_matrix[i][j] > threshold:
                G.add_edge(i, j,
                           weight=float(sim_matrix[i][j]),
                           cross_topic=not same_topic)

    return G, all_papers


def plot_citation_network(queries, papers_per_query=5):
    G, all_papers = build_citation_network(queries, papers_per_query)
    colors_list = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

    plt.figure(figsize=(16, 12))
    pos = nx.spring_layout(G, seed=42, k=1.5)

    node_colors = [G.nodes[n]['color'] for n in G.nodes]
    intra_edges = [(u,v) for u,v,d in G.edges(data=True) if not d.get('cross_topic')]
    cross_edges = [(u,v) for u,v,d in G.edges(data=True) if d.get('cross_topic')]

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500, alpha=0.85)
    nx.draw_networkx_edges(G, pos, edgelist=intra_edges, width=1.5, alpha=0.4, edge_color='gray')
    nx.draw_networkx_edges(G, pos, edgelist=cross_edges, width=2.5, alpha=0.8,
                           edge_color='black', style='dashed')
    labels = {n: G.nodes[n]['title'] for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels, font_size=6)

    # Legend
    from matplotlib.patches import Patch
    legend = [Patch(color=colors_list[i], label=q[:30]) for i, q in enumerate(queries)]
    plt.legend(handles=legend, loc='upper left', fontsize=9, title='Topics')

    plt.title('Citation Network — Topic Clusters\n(Dashed = Cross-topic connections)',
              fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('outputs/04_citation_network.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    print(f'Cross-topic connections: {len(cross_edges)}')

# Test with 3 topics
topics = [
    'transformer attention mechanism',
    'reinforcement learning policy gradient',
    'generative adversarial networks image synthesis'
]
plot_citation_network(topics, papers_per_query=5)

## 8. ⚖️ Compare 2 Papers Side-by-Side (UNIQUE)
### No one did this in Streamlit!

In [ ]:
def compare_papers(query1, query2):
    """
    Find best paper for each query and compare them side by side.
    Shows: summaries, keywords, and similarity score between the two papers.
    """
    r1 = semantic_search(query1, top_k=1, mmr=False)[0]
    r2 = semantic_search(query2, top_k=1, mmr=False)[0]

    # Embeddings for cross-paper similarity
    emb1 = embeddings[r1['idx']].reshape(1, -1)
    emb2 = embeddings[r2['idx']].reshape(1, -1)
    cross_sim = cosine_similarity(emb1, emb2)[0][0]

    # Summarize
    sum1 = summarize(r1['abstract'])
    sum2 = summarize(r2['abstract'])

    # Keywords
    kw1 = extract_keywords(r1['abstract'])
    kw2 = extract_keywords(r2['abstract'])

    # Common keywords
    common_kw = set(kw1) & set(kw2)

    print('='*70)
    print('  PAPER COMPARISON')
    print('='*70)
    print(f'\n📄 PAPER A: {r1["title"]}')
    print(f'   Relevance Score : {r1["score"]:.4f}')
    print(f'   Summary         : {sum1}')
    print(f'   Keywords        : {kw1}')

    print(f'\n📄 PAPER B: {r2["title"]}')
    print(f'   Relevance Score : {r2["score"]:.4f}')
    print(f'   Summary         : {sum2}')
    print(f'   Keywords        : {kw2}')

    print(f'\n🔗 Cross-Paper Similarity : {cross_sim:.4f}')
    print(f'   Common Keywords        : {common_kw if common_kw else "None"}')
    print('='*70)

    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, title, kws, color in [
        (axes[0], r1['title'][:50], kw1, '#3498db'),
        (axes[1], r2['title'][:50], kw2, '#e74c3c')
    ]:
        ax.barh(range(len(kws)), [1]*len(kws), color=color, alpha=0.7)
        ax.set_yticks(range(len(kws)))
        ax.set_yticklabels(kws, fontsize=10)
        ax.set_title(title, fontsize=9, fontweight='bold', wrap=True)
        ax.set_xticks([])

    plt.suptitle(f'Paper Comparison — Cross Similarity: {cross_sim:.3f}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/05_paper_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {'paper_a': r1, 'paper_b': r2, 'similarity': cross_sim,
            'summary_a': sum1, 'summary_b': sum2,
            'keywords_a': kw1, 'keywords_b': kw2, 'common': list(common_kw)}

# Test
comparison = compare_papers(
    'BERT language model pre-training',
    'GPT generative language model'
)

## 9. 📄 PDF Export (UNIQUE)
### Export search results as downloadable PDF report

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from datetime import datetime

def export_to_pdf(query, results, summaries=None, keywords_list=None,
                  filename='outputs/search_results.pdf'):
    """
    Export search results to a formatted PDF report.
    """
    doc = SimpleDocTemplate(filename, pagesize=letter,
                            rightMargin=0.75*inch, leftMargin=0.75*inch,
                            topMargin=1*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()

    # Custom styles
    title_style = ParagraphStyle('CustomTitle', parent=styles['Title'],
                                  fontSize=18, textColor=colors.HexColor('#1F4E79'),
                                  spaceAfter=12)
    heading_style = ParagraphStyle('PaperTitle', parent=styles['Heading2'],
                                    fontSize=12, textColor=colors.HexColor('#2E75B6'),
                                    spaceAfter=6)
    body_style = ParagraphStyle('Body', parent=styles['Normal'],
                                 fontSize=10, spaceAfter=6, leading=14)
    meta_style = ParagraphStyle('Meta', parent=styles['Normal'],
                                 fontSize=9, textColor=colors.gray, spaceAfter=4)

    story = []

    # Header
    story.append(Paragraph('AI Research Paper Intelligence System', title_style))
    story.append(Paragraph('CBSOT Summer Internship 2026', styles['Normal']))
    story.append(Spacer(1, 0.1*inch))
    story.append(HRFlowable(width='100%', thickness=2, color=colors.HexColor('#2E75B6')))
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph(f'Search Query: <b>{query}</b>', styles['Normal']))
    story.append(Paragraph(f'Results Found: {len(results)} papers', meta_style))
    story.append(Paragraph(f'Generated: {datetime.now().strftime("%d %B %Y, %H:%M")}', meta_style))
    story.append(Spacer(1, 0.2*inch))

    # Results
    for i, r in enumerate(results):
        story.append(Paragraph(f'{i+1}. {r["title"]}', heading_style))
        story.append(Paragraph(f'Relevance Score: {r["score"]:.4f}', meta_style))

        if summaries and i < len(summaries):
            story.append(Paragraph(f'<b>Summary:</b> {summaries[i]}', body_style))

        if keywords_list and i < len(keywords_list):
            kw_text = ', '.join(keywords_list[i])
            story.append(Paragraph(f'<b>Keywords:</b> {kw_text}', body_style))

        abstract_short = r['abstract'][:400] + '...' if len(r['abstract']) > 400 else r['abstract']
        story.append(Paragraph(f'<b>Abstract:</b> {abstract_short}', body_style))
        story.append(HRFlowable(width='100%', thickness=0.5, color=colors.lightgrey))
        story.append(Spacer(1, 0.1*inch))

    doc.build(story)
    print(f'✅ PDF exported: {filename}')
    return filename

# Test
query = 'attention mechanism transformer NLP'
results = semantic_search(query, top_k=5)
summaries = [summarize(r['abstract']) for r in results]
keywords_list = [extract_keywords(r['abstract']) for r in results]
export_to_pdf(query, results, summaries, keywords_list)

## 10. 🔖 Bookmarks & 📜 Search History (UNIQUE)
### Persistent across session via pickle

In [ ]:
import json
from datetime import datetime

BOOKMARKS_FILE = 'outputs/bookmarks.json'
HISTORY_FILE   = 'outputs/search_history.json'

def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return []

def save_json(path, data):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def add_bookmark(paper):
    bookmarks = load_json(BOOKMARKS_FILE)
    entry = {
        'title': paper['title'],
        'abstract': paper['abstract'][:300],
        'score': paper['score'],
        'saved_at': datetime.now().isoformat()
    }
    if not any(b['title'] == entry['title'] for b in bookmarks):
        bookmarks.append(entry)
        save_json(BOOKMARKS_FILE, bookmarks)
        print(f'✅ Bookmarked: {paper["title"][:60]}')
    else:
        print('Already bookmarked!')

def add_to_history(query, num_results):
    history = load_json(HISTORY_FILE)
    history.insert(0, {
        'query': query,
        'results': num_results,
        'timestamp': datetime.now().isoformat()
    })
    history = history[:50]  # Keep last 50
    save_json(HISTORY_FILE, history)

def show_bookmarks():
    bookmarks = load_json(BOOKMARKS_FILE)
    print(f'📚 Saved Bookmarks ({len(bookmarks)} total):')
    for i, b in enumerate(bookmarks):
        print(f'  {i+1}. {b["title"][:60]}')
        print(f'     Saved: {b["saved_at"][:10]}')

def show_history():
    history = load_json(HISTORY_FILE)
    print(f'🕐 Search History ({len(history)} searches):')
    for h in history[:10]:
        print(f'  [{h["timestamp"][:10]}] "{h["query"]}" → {h["results"]} results')

# Test
results = semantic_search('deep learning image classification', top_k=3)
add_to_history('deep learning image classification', len(results))
add_bookmark(results[0])
add_bookmark(results[1])

results2 = semantic_search('recurrent neural networks LSTM', top_k=3)
add_to_history('recurrent neural networks LSTM', len(results2))
add_bookmark(results2[0])

print()
show_bookmarks()
print()
show_history()

## 11. Full Pipeline Test

In [ ]:
print('=== FULL PIPELINE TEST ===')
query = 'contrastive learning self-supervised representation'

# Search
results = semantic_search(query, top_k=5, mmr=True)
add_to_history(query, len(results))

print(f'Query: "{query}"')
print(f'Found: {len(results)} papers\n')

for i, r in enumerate(results):
    print(f'{i+1}. {r["title"]}')
    print(f'   Score   : {r["score"]:.4f}')
    summary = summarize(r['abstract'])
    print(f'   Summary : {summary[:150]}...')
    kws = extract_keywords(r['abstract'], top_n=5)
    print(f'   Keywords: {kws}')
    print()

# Export to PDF
export_to_pdf(query, results)

print('\n✅ Full pipeline working!')
print('➡️  Next: Run the Streamlit app!')